# Datos y modelo baseline en Keras

Carga, partición y normalización de MNIST; se inspecciona el balance de clases y se entrena un modelo base.


# Preparación de datos MNIST

Este bloque se incluye en cada notebook para que pueda ejecutarse de forma independiente.

In [ ]:
# Librerías, semilla y carga de MNIST
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)

(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train, x_dev, y_train, y_dev = train_test_split(
    x_train_full, y_train_full, test_size=0.20, random_state=SEED, stratify=y_train_full
)
x_train = x_train.astype("float32") / 255.0
x_dev = x_dev.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
print("Train:", x_train.shape, "Dev:", x_dev.shape, "Test:", x_test.shape)


In [ ]:
def distribucion_clases(y, nombre):
    conteos = np.bincount(y, minlength=10)
    tabla = pd.DataFrame({"digito": np.arange(10), "cantidad": conteos,
                         "porcentaje": np.round(conteos / len(y) * 100, 2)})
    print(nombre)
    display(tabla)
    return tabla

dist_train = distribucion_clases(y_train, "TRAIN")
dist_dev = distribucion_clases(y_dev, "DEV")
dist_test = distribucion_clases(y_test, "TEST")


In [ ]:
def crear_modelo_baseline(units=128, optimizer=None):
    modelo = keras.Sequential([layers.Input(shape=(28, 28)), layers.Flatten(),
        layers.Dense(units, activation="relu"), layers.Dense(10, activation="softmax")])
    if optimizer is None:
        optimizer = keras.optimizers.Adam(learning_rate=0.001)
    modelo.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return modelo

modelo_base = crear_modelo_baseline()
modelo_base.summary()
hist_base = modelo_base.fit(x_train, y_train, validation_data=(x_dev, y_dev),
                           epochs=5, batch_size=64, verbose=1)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_base.history["loss"], label="Train")
axes[0].plot(hist_base.history["val_loss"], label="Dev")
axes[0].set(title="Baseline - Loss", xlabel="Época", ylabel="Loss")
axes[1].plot(hist_base.history["accuracy"], label="Train")
axes[1].plot(hist_base.history["val_accuracy"], label="Dev")
axes[1].set(title="Baseline - Accuracy", xlabel="Época", ylabel="Accuracy")
for ax in axes: ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()
